# Devoir 3 — Implémentation et comparaison des architectures RAG

## Sujet personnel : Génération de règles SNORT

Ce notebook implémente un prototype complet pour comparer plusieurs architectures RAG dans un contexte de cybersécurité. L'objectif est de générer une règle SNORT à partir d'une description naturelle d'un comportement réseau suspect.

Architectures comparées :

- Baseline sans RAG
- RAG classique
- RAG avec re-ranking
- RAG hybride dense + BM25
- Multi-hop RAG
- Graph RAG
- Agentic RAG

**Important :** aucune API LLM externe n'est utilisée. La génération est locale et template-based, ancrée dans les documents récupérés.


## 1. Préparation et imports

Le projet contient un dataset personnel synthétique dans `data/`, les modules Python dans `src/` et les résultats dans `outputs/`.


In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src.data_generator import save_dataset
from src.rag_snort import SnortRAGEngine
from src.metrics import evaluate_prediction

DATA_DIR = PROJECT_ROOT / 'data'
OUTPUTS_DIR = PROJECT_ROOT / 'outputs'
OUTPUTS_DIR.mkdir(exist_ok=True)
print('Project root:', PROJECT_ROOT)


## 2. Génération / chargement du dataset

Le corpus est structuré en CSV/JSON. Chaque ligne contient une description d'attaque, un label, un extrait de log simulé, une règle SNORT attendue et une explication.


In [ ]:
if not (DATA_DIR / 'snort_knowledge_base.csv').exists():
    save_dataset(DATA_DIR, n_rows=160, n_queries=32)

kb = pd.read_csv(DATA_DIR / 'snort_knowledge_base.csv')
queries = pd.read_csv(DATA_DIR / 'snort_test_queries.csv')

print('Documents:', len(kb))
print('Test queries:', len(queries))
kb.head(3)


In [ ]:
kb[['attack_family', 'attack_type', 'protocol', 'destination_port', 'severity']].head()


## 3. Exploration rapide du dataset


In [ ]:
family_counts = kb['attack_family'].value_counts()
family_counts


In [ ]:
plt.figure(figsize=(9, 4))
family_counts.plot(kind='bar')
plt.title('Distribution des familles d'attaques dans le dataset')
plt.xlabel('Famille d'attaque')
plt.ylabel('Nombre de documents')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()


## 4. Construction du moteur RAG

Le moteur contient :

- représentation sparse TF-IDF ;
- représentation dense locale avec SVD + normalisation ;
- BM25 local ;
- FAISS si disponible, sinon cosine similarity ;
- graphe simple famille/type/protocole/port/keywords.


In [ ]:
engine = SnortRAGEngine(kb)
print('Dense embedding shape:', engine.dense.shape)
print('FAISS available:', engine.faiss_index is not None)


## 5. Test manuel des architectures


In [ ]:
sample_query = 'Detect a TCP SYN port scan targeting a web server on port 80'
for arch in ['baseline', 'rag_classic', 'rag_rerank', 'rag_hybrid', 'multi_hop', 'graph_rag', 'agentic_rag']:
    pred = engine.run_architecture(arch, sample_query, k=5)
    print('
ARCHITECTURE:', arch)
    print('Retrieved:', pred.get('retrieved_ids'))
    print('Rule:', pred.get('generated_rule'))


## 6. Expérimentation complète

On évalue chaque architecture sur toutes les requêtes de test.


In [ ]:
ARCHITECTURES = ['baseline', 'rag_classic', 'rag_rerank', 'rag_hybrid', 'multi_hop', 'graph_rag', 'agentic_rag']

rows = []
predictions = []
for _, qrow in queries.iterrows():
    expected = qrow.to_dict()
    for arch in ARCHITECTURES:
        pred = engine.run_architecture(arch, qrow['query'], k=5)
        metrics = evaluate_prediction(pred, expected, k=5)
        rows.append({
            'query_id': qrow['query_id'],
            'architecture': arch,
            'query': qrow['query'],
            **metrics,
            'expected_doc_id': qrow['expected_doc_id'],
            'retrieved_ids': ';'.join(pred.get('retrieved_ids', [])),
            'generated_rule': pred.get('generated_rule'),
            'expected_rule': qrow['expected_rule']
        })
        predictions.append({'query_id': qrow['query_id'], 'architecture': arch, **pred})

detailed = pd.DataFrame(rows)
detailed.head()


## 7. Tableau comparatif des résultats


In [ ]:
summary = detailed.groupby('architecture').agg({
    'precision_at_3': 'mean',
    'recall_at_3': 'mean',
    'recall_at_5': 'mean',
    'mrr': 'mean',
    'ndcg_at_5': 'mean',
    'family_accuracy': 'mean',
    'type_accuracy': 'mean',
    'protocol_accuracy': 'mean',
    'port_accuracy': 'mean',
    'snort_syntax_valid': 'mean',
    'rule_token_jaccard': 'mean',
    'hallucination_flag': 'mean'
}).reset_index().sort_values(['recall_at_5', 'family_accuracy', 'rule_token_jaccard'], ascending=False)

summary


In [ ]:
detailed.to_csv(OUTPUTS_DIR / 'detailed_results.csv', index=False)
summary.to_csv(OUTPUTS_DIR / 'comparison_summary.csv', index=False)
print('Saved results in', OUTPUTS_DIR)


## 8. Visualisation des embeddings avec t-SNE

Cette visualisation permet d'observer si les documents de familles proches se regroupent dans l'espace vectoriel.


In [ ]:
from sklearn.manifold import TSNE

# Pour accélérer l'exécution, on peut échantillonner si le corpus devient grand.
emb = engine.dense
perplexity = min(20, max(5, len(kb) // 10))
coords = TSNE(
    n_components=2,
    random_state=42,
    perplexity=perplexity,
    init='random',
    learning_rate='auto',
    max_iter=500
).fit_transform(emb)

plt.figure(figsize=(9, 6))
for family in sorted(kb['attack_family'].unique()):
    mask = kb['attack_family'] == family
    plt.scatter(coords[mask, 0], coords[mask, 1], label=family, s=25, alpha=0.8)
plt.title('Visualisation t-SNE des embeddings')
plt.xlabel('t-SNE 1')
plt.ylabel('t-SNE 2')
plt.legend(fontsize=8)
plt.tight_layout()
plt.savefig(OUTPUTS_DIR / 'tsne_embeddings.png', dpi=160)
plt.show()


## 9. Interface Gradio

L'interface est disponible dans `app_gradio.py`. Elle permet de saisir une description d'attaque, de choisir l'architecture et de voir la règle générée.

Exécution :

```bash
python app_gradio.py
```


## 10. Analyse critique finale

### Quelle architecture est la plus performante ?

D'après le tableau comparatif, les architectures qui combinent plusieurs signaux de récupération, notamment **RAG hybride** et **Agentic RAG**, obtiennent généralement les meilleurs résultats. Elles profitent à la fois des similarités lexicales (ports, protocoles, mots-clés SNORT) et d'une représentation dense locale.

### Quelle architecture est la plus robuste ?

L'**Agentic RAG** est la plus robuste parce qu'elle ajoute une étape de décision et de validation. Si la requête est trop courte ou si la règle générée semble invalide, elle peut changer de stratégie.

### Quelle architecture est la plus adaptée au projet ?

Pour le PFM, l'architecture la plus adaptée comme base est le **RAG hybride**, car SNORT repose fortement sur des éléments exacts : protocoles, ports, flags, contenus, signatures et mots-clés. Le BM25/TF-IDF aide à retrouver ces éléments, tandis que la partie dense améliore la flexibilité.

### Quelle architecture produit le plus d'hallucinations ?

La **baseline sans RAG** est la plus risquée parce qu'elle génère sans document récupéré. Les architectures RAG limitent ce risque en ancrant la réponse dans les exemples retrouvés.

### Limites

- Le dataset est synthétique.
- La génération est template-based pour respecter l'interdiction des API LLM.
- Une validation humaine experte reste nécessaire avant toute utilisation réelle d'une règle SNORT.
- Pour le PFM, il faudra enrichir la base, ajouter le dashboard et la possibilité d'ajouter des PDF comme base de connaissance.
